[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/15_agent_v3.ipynb)

# 15. Agent v3

**Part 3: Context** · about 30 minutes

Part 3 gave the agent ways to make the most of limited context: measure it (11), shrink it (12), find code instead of reading it (13), and carry knowledge between runs (14). Now they all go into one agent, on the shop project from notebook 13, with a bug report from an unhappy customer:

> "I've been shopping here for five years, and my bill for 10 masala teas was higher than my friend's, who signed up last week. Aren't loyal customers supposed to get a discount?"

Compared with agent v2:

| | Agent v2 (notebook 10) | Agent v3 (this notebook) |
|---|---|---|
| Finding code | `find` (substring) | `grep`, `outline`, `find_definition`, `find_references` |
| Reading | Line ranges | Line ranges, results capped by a token budget |
| Long runs | History grows forever | Old tool results cleared when over budget |
| Between runs | Starts from zero | Loads and updates a memory file |
| Visibility | Trace | Trace plus a context report after every step |

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import ast
import difflib
import json
import os
import re
import shlex
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from datetime import date
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. The project, with a bug

The shop from notebook 13, with one character changed in `final_price`. The memory file already has one note from an earlier run, as it would on a project the agent has worked on before.

In [3]:
SHOP = {
    "shop/__init__.py": '"""A small online shop: products, orders, pricing and receipts."""\n',
    "shop/models.py": '''\
from dataclasses import dataclass, field


@dataclass
class Product:
    sku: str
    name: str
    price: float  # in rupees


@dataclass
class Customer:
    name: str
    email: str
    years: int = 0  # how long they've been a customer


@dataclass
class Order:
    customer: Customer
    lines: list = field(default_factory=list)  # (Product, quantity) pairs

    def add(self, product, quantity=1):
        if quantity < 1:
            raise ValueError("quantity must be at least 1")
        self.lines.append((product, quantity))
''',
    "shop/pricing.py": '''\
"""Prices, discounts and tax."""

GST_RATE = 0.18


def subtotal(order):
    """Sum of price times quantity for every line."""
    return sum(product.price * quantity for product, quantity in order.lines)


def loyalty_discount(amount, years):
    """2% off for every year as a customer, capped at 10%."""
    rate = min(years * 2, 10)
    return round(amount * rate / 100, 2)


def coupon_discount(amount, code):
    """Flat discounts for known coupon codes, never more than the amount."""
    coupons = {"WELCOME50": 50, "DIWALI200": 200}
    return min(coupons.get(code, 0), amount)


def final_price(order, coupon=None):
    """What the customer pays, including GST."""
    amount = subtotal(order)
    amount = amount - loyalty_discount(amount, order.customer.years)
    if coupon:
        amount -= coupon_discount(amount, coupon)
    return round(amount * (1 + GST_RATE), 2)
''',
    "shop/inventory.py": '''\
"""Stock levels and reservations."""

STOCK = {}


def set_stock(sku, quantity):
    STOCK[sku] = quantity


def available(sku):
    return STOCK.get(sku, 0)


def reserve(order):
    """Take stock for every line in the order, or nothing if any line is short."""
    for product, quantity in order.lines:
        if available(product.sku) < quantity:
            raise ValueError(f"not enough stock for {product.name}")
    for product, quantity in order.lines:
        STOCK[product.sku] -= quantity


def release(order):
    """Put an order's stock back, for example after a cancellation."""
    for product, quantity in order.lines:
        STOCK[product.sku] = available(product.sku) + quantity
''',
    "shop/orders.py": '''\
"""Placing and cancelling orders."""

from shop import inventory, notifications
from shop.pricing import final_price

PLACED = []


def place_order(order, coupon=None):
    inventory.reserve(order)
    total = final_price(order, coupon)
    PLACED.append((order, total))
    notifications.send_receipt(order, total)
    return total


def cancel_order(order):
    for index, (placed, _) in enumerate(PLACED):
        if placed is order:
            PLACED.pop(index)
            inventory.release(order)
            return True
    return False
''',
    "shop/notifications.py": '''\
"""Customer messages. Returns text instead of sending email, to keep things testable."""

from shop.utils import format_rupees

OUTBOX = []


def send_receipt(order, total):
    lines = [f"Hi {order.customer.name}, thanks for your order!"]
    for product, quantity in order.lines:
        lines.append(f"  {quantity} x {product.name}")
    lines.append(f"Total paid: {format_rupees(total)}")
    message = "\\n".join(lines)
    OUTBOX.append((order.customer.email, message))
    return message
''',
    "shop/reports.py": '''\
"""Numbers for the shop owner."""

from shop.pricing import final_price, loyalty_discount, subtotal


def daily_revenue(placed):
    return round(sum(total for _, total in placed), 2)


def loyalty_savings(orders):
    """How much loyal customers saved before tax, in total."""
    return round(sum(loyalty_discount(subtotal(o), o.customer.years) for o in orders), 2)


def biggest_order(orders):
    return max(orders, key=final_price, default=None)
''',
    "shop/utils.py": '''\
"""Small helpers."""


def format_rupees(amount):
    """Format with Indian digit grouping, like Rs 1,23,456.50."""
    rupees, paise = f"{amount:.2f}".split(".")
    if len(rupees) > 3:
        head, tail = rupees[:-3], rupees[-3:]
        groups = []
        while len(head) > 2:
            groups.insert(0, head[-2:])
            head = head[:-2]
        rupees = ",".join([head] + groups + [tail]) if head else ",".join(groups + [tail])
    return f"Rs {rupees}.{paise}"
''',
    "tests/test_pricing.py": '''\
import unittest

from shop.models import Customer, Order, Product
from shop.pricing import coupon_discount, final_price, loyalty_discount, subtotal

TEA = Product("T1", "Masala tea", 100.0)


class PricingTests(unittest.TestCase):
    def test_subtotal(self):
        order = Order(Customer("Asha", "asha@example.com"))
        order.add(TEA, 3)
        self.assertEqual(subtotal(order), 300.0)

    def test_loyalty_is_capped(self):
        self.assertEqual(loyalty_discount(1000, 2), 40.0)
        self.assertEqual(loyalty_discount(1000, 20), 100.0)

    def test_loyal_customers_pay_less(self):
        new, loyal = Order(Customer("A", "a@example.com", 0)), Order(Customer("B", "b@example.com", 5))
        new.add(TEA, 10)
        loyal.add(TEA, 10)
        self.assertLess(final_price(loyal), final_price(new))
        self.assertEqual(final_price(loyal), 1062.0)

    def test_coupon_never_exceeds_amount(self):
        self.assertEqual(coupon_discount(30, "WELCOME50"), 30)
''',
    "tests/test_utils.py": '''\
import unittest

from shop.utils import format_rupees


class FormatTests(unittest.TestCase):
    def test_small(self):
        self.assertEqual(format_rupees(950), "Rs 950.00")

    def test_indian_grouping(self):
        self.assertEqual(format_rupees(123456.5), "Rs 1,23,456.50")
''',
}

workspace = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()
for path, text in SHOP.items():
    (workspace / path).parent.mkdir(parents=True, exist_ok=True)
    (workspace / path).write_text(text)

CORRECT = SHOP["shop/pricing.py"]
BUGGY = CORRECT.replace("amount = amount - loyalty_discount(", "amount = amount + loyalty_discount(")
assert BUGGY != CORRECT
(workspace / "shop/pricing.py").write_text(BUGGY)

MEMORY_FILE = workspace / ".agent" / "memory.md"
MEMORY_FILE.parent.mkdir()
MEMORY_FILE.write_text("- Money: prices are in rupees, GST is 18% (saved 2026-11-01)\n")

print(f"{len(SHOP)} files. Memory: {MEMORY_FILE.read_text().strip()}")

10 files. Memory: - Money: prices are in rupees, GST is 18% (saved 2026-11-01)


## 2. The toolbox

All of this comes from earlier notebooks: files from 06 and 10, commands from 07 to 10, search from 13, memory from 14 and budgets from 11. It's collected here so this notebook runs on its own. Each tool is unchanged apart from being shorter, so skip ahead to section 3 if you've done Parts 2 and 3.

In [4]:
def estimate_tokens(text):
    return max(1, len(text) // 4)


RESULT_LIMIT = 800  # tokens per tool result (notebook 11)


def cap(text, limit=RESULT_LIMIT):
    if estimate_tokens(text) <= limit:
        return text
    half = limit * 2
    return f"{text[:half]}\n[... cut: about {estimate_tokens(text):,} tokens, limit {limit:,}. Narrow your request.]\n{text[-half:]}"


def safe_path(path):
    full = (workspace / path).resolve()
    if not full.is_relative_to(workspace):
        raise PermissionError(f"{path!r} is outside the workspace")
    return full


# --- Files ---------------------------------------------------------------------------

def read_file(path, start=1, end=None):
    lines = safe_path(path).read_text().splitlines()
    end = len(lines) if end is None else min(int(end), len(lines))
    start = int(start)
    if not 1 <= start <= max(len(lines), 1):
        return f"Error: start must be between 1 and {len(lines)}"
    shown = "\n".join(f"{n:>4} | {line}" for n, line in enumerate(lines[start - 1 : end], start))
    return f"{path}: lines {start}-{end} of {len(lines)}\n{shown}"


def edit_file(path, old, new):
    file = safe_path(path)
    text = file.read_text()
    if old == new:
        return "Error: old and new are identical."
    if re.search(r"^\s*\d+ \| ", old, re.MULTILINE):
        return "Error: old contains line numbers like '  12 | '. Copy only the code after the bar."
    count = text.count(old)
    if count != 1:
        hint = "not found" if count == 0 else f"found {count} times; include more surrounding lines"
        return f"Error: old was {hint}."
    line = text[: text.index(old)].count("\n") + 1
    file.write_text(text.replace(old, new, 1))
    return f"Edited {path}.\n" + read_file(path, max(1, line - 2), line + new.count("\n") + 2)


# --- Search ----------------------------------------------------------------------------

def python_files():
    for file in sorted(workspace.rglob("*.py")):
        yield file.relative_to(workspace).as_posix(), file.read_text()


def grep(pattern):
    try:
        regex = re.compile(pattern, re.IGNORECASE)
    except re.error as error:
        return f"Error: invalid regular expression ({error})."
    hits = [f"{p}:{n}: {line.strip()}" for p, text in python_files()
            for n, line in enumerate(text.splitlines(), 1) if regex.search(line)]
    return "\n".join(hits[:40]) or f"No matches for {pattern!r}"


def outline(path):
    module = ast.parse(safe_path(path).read_text())
    rows = [f"{n.lineno}-{n.end_lineno}  {'class' if isinstance(n, ast.ClassDef) else 'def'} {n.name}"
            for n in module.body if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
    return f"{path}\n" + "\n".join(rows)


def find_definition(name):
    found = []
    for path, text in python_files():
        for node in ast.walk(ast.parse(text)):
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name == name:
                found.append(f"{path}:{node.lineno}-{node.end_lineno}: {text.splitlines()[node.lineno - 1].strip()}")
    return "\n".join(found) or f"No definition of {name!r} found."


def find_references(name):
    found = []
    for path, text in python_files():
        module, lines = ast.parse(text), text.splitlines()
        functions = [n for n in ast.walk(module) if isinstance(n, ast.FunctionDef)]
        for node in ast.walk(module):
            if isinstance(node, ast.ImportFrom) and any(a.name == name for a in node.names):
                found.append(f"{path}:{node.lineno} (import): {lines[node.lineno - 1].strip()}")
            elif (isinstance(node, ast.Name) and node.id == name) or (isinstance(node, ast.Attribute) and node.attr == name):
                inside = [f for f in functions if f.lineno <= node.lineno <= f.end_lineno]
                where = max(inside, key=lambda f: f.lineno).name if inside else "module level"
                found.append(f"{path}:{node.lineno} (in {where}): {lines[node.lineno - 1].strip()}")
    return "\n".join(sorted(set(found))) or f"No uses of {name!r} found."


# --- Memory -----------------------------------------------------------------------------

def load_memory():
    return MEMORY_FILE.read_text() if MEMORY_FILE.exists() else ""


def remember(note):
    note = " ".join(note.split())
    if len(note) > 300:
        return "Error: keep notes under 300 characters."
    label = note.split(":")[0].strip().lower() if ":" in note else None
    lines = [l for l in load_memory().splitlines() if l.startswith("- ")]
    if label:
        lines = [l for l in lines if l[2:].split(" (")[0].split(":")[0].strip().lower() != label]
    lines.append(f"- {note} (saved {date.today().isoformat()})")
    MEMORY_FILE.write_text("\n".join(lines) + "\n")
    print(f"     MEMORY WRITE: {note}")
    return f"Saved. Memory now has {len(lines)} notes."

In [5]:
# --- Commands: permissions and sandbox (notebooks 08 to 10) ------------------------------

OPERATORS = {";", "&&", "||", "|", "&", ">", ">>", "<", "(", ")"}
READ_ONLY = {"ls", "cat", "head", "tail", "wc", "grep", "find", "pwd", "echo", "diff"}
NEVER = {"sudo", "su", "curl", "wget", "ssh", "scp", "nc", "dd", "mkfs", "shutdown", "reboot", "chmod", "chown"}
HAS_LIMITS = os.name == "posix"
if HAS_LIMITS:
    import resource
AUDIT = []


def decide(command):
    try:
        lexer = shlex.shlex(command, posix=True, punctuation_chars=True)
        lexer.whitespace_split = True
        tokens = list(lexer)
    except ValueError:
        return "ask", "couldn't parse the command"
    if not tokens:
        return "deny", "empty command"
    programs = [tokens[0]] + [tokens[i + 1] for i, t in enumerate(tokens[:-1]) if t in OPERATORS]
    if {Path(p).name for p in programs} & NEVER:
        return "deny", "uses a program that's never allowed"
    if OPERATORS & set(tokens) or "`" in command or "$(" in command:
        return "ask", "compound command"
    program = Path(tokens[0]).name
    if program == "find" and {"-delete", "-exec", "-execdir", "-ok"} & set(tokens):
        return "ask", "find that changes or runs things"
    if program in READ_ONLY or tokens[:3] == ["python", "-m", "unittest"]:
        return "allow", "safe command"
    if program in ("python", "python3") and len(tokens) > 1 and tokens[1].endswith(".py"):
        return "allow", "project script, sandboxed"
    return "ask", f"{program} isn't on the allow list"


def apply_limits():
    resource.setrlimit(resource.RLIMIT_CPU, (10, 11))
    resource.setrlimit(resource.RLIMIT_FSIZE, (20 * 1024 * 1024,) * 2)


def run_command(command):
    decision, reason = decide(command)
    AUDIT.append((command, decision))
    if decision == "deny":
        return f"Denied by policy: {reason}. Don't look for a way around this."
    if decision == "ask":  # no human in this notebook; see notebook 08 for interactive approval
        return f"Denied: this needs approval ({reason}) and no one is here to approve it. Don't retry it."
    shell_command = re.sub(r"(?<![\w./-])python3?(?=\s|$)", lambda _: shlex.quote(sys.executable), command)
    env = {k: os.environ[k] for k in ("PATH", "SYSTEMROOT", "LANG") if k in os.environ}
    env["HOME"] = str(workspace)
    options = dict(shell=True, cwd=workspace, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if HAS_LIMITS:
        options.update(start_new_session=True, preexec_fn=apply_limits)
    process = subprocess.Popen(shell_command, **options)
    try:
        output, _ = process.communicate(timeout=30)
        status = f"exit code: {process.returncode}"
    except subprocess.TimeoutExpired:
        if HAS_LIMITS:
            os.killpg(process.pid, signal.SIGKILL)
        else:
            subprocess.run(["taskkill", "/T", "/F", "/PID", str(process.pid)], capture_output=True)
        output, _ = process.communicate()
        status = "stopped: took longer than 30 seconds"
    return f"{status}\n{output.replace(str(workspace), '.')}"

In [6]:
def tool(tool_name, description, /, **properties):
    required = [k for k, v in properties.items() if not v.startswith("(optional)")]
    return {"type": "function", "function": {"name": tool_name, "description": description, "parameters": {
        "type": "object", "required": required,
        "properties": {k: {"type": "integer" if k in ("start", "end") else "string", "description": v}
                       for k, v in properties.items()}}}}


TOOLS = [
    tool("run_command", "Run a shell command in the project, sandboxed. Output starts with the exit code.",
         command="The command"),
    tool("grep", "Search all Python files with a case-insensitive regex. Returns file:line: text.", pattern="Regex"),
    tool("outline", "List a file's classes and functions with line ranges.", path="File path"),
    tool("find_definition", "Find where a function or class is defined.", name="Exact name"),
    tool("find_references", "Find every use of a function or class, and the function each use is in.", name="Exact name"),
    tool("read_file", "Read a file with line numbers.", path="File path",
         start="(optional) first line", end="(optional) last line"),
    tool("edit_file", "Replace text that appears exactly once in a file. No line numbers in old.",
         path="File path", old="Exact text to replace", new="Replacement"),
    tool("remember", "Save a short fact for future runs, as 'Label: fact'. Only durable facts.", note="The note"),
]
REGISTRY = {"run_command": run_command, "grep": grep, "outline": outline, "find_definition": find_definition,
            "find_references": find_references, "read_file": read_file, "edit_file": edit_file, "remember": remember}
print(f"{len(TOOLS)} tools:", ", ".join(REGISTRY))

8 tools: run_command, grep, outline, find_definition, find_references, read_file, edit_file, remember


## 3. The loop, with a context budget

Two things are new in the loop itself:

1. **A context report after every step,** a single line showing how big the next request will be. When something goes wrong in a long run, this line tells you whether context was the problem.
2. **Clearing over budget.** If the history passes `CONTEXT_BUDGET`, old tool results are cleared (notebook 12). We use clearing rather than summarising here because a bug fix rarely needs the content of a test run from ten steps ago, and clearing is free. Results from the last two tool calls are always kept.

In [7]:
CONTEXT_BUDGET = 6000  # tokens; small on purpose so you can see it work on a real model

SYSTEM_PROMPT = """You are a careful coding agent working on a Python project you may have seen before.

1. Reproduce the problem first: run the tests or the command from the report.
2. Use the search tools to find the code involved. Read only the lines you need.
3. Fix it with edit_file, changing as little as possible. Don't edit tests.
4. Run the tests again. You're done only when they pass.
5. If you learned something that will save time on the next run, save it with remember.

Finish with a short summary of the cause and the fix. Don't claim anything you didn't see in a tool result."""


def context_size(messages):
    return sum(estimate_tokens((m.get("content") or "") + json.dumps(m.get("tool_calls") or [])) for m in messages) \
        + estimate_tokens(json.dumps(TOOLS))


def clear_old_results(messages, keep_last=2):
    tool_indexes = [i for i, m in enumerate(messages) if m["role"] == "tool"]
    old = set(tool_indexes[:-keep_last])
    return [{**m, "content": "[cleared to save space]"} if i in old and m["content"] != "[cleared to save space]"
            else m for i, m in enumerate(messages)]


def run_agent(task, max_steps=25):
    system = SYSTEM_PROMPT
    memory = load_memory()
    if memory:
        system += f"\n\nWhat you remember about this project:\n{memory}"
    messages = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    stats = {"model_calls": 0, "tool_calls": 0, "input_tokens": 0, "peak_context": 0, "clearings": 0}

    for step in range(1, max_steps + 1):
        size = context_size(messages)
        if size > CONTEXT_BUDGET:
            messages = clear_old_results(messages)
            stats["clearings"] += 1
            print(f"     context {size:,} > budget {CONTEXT_BUDGET:,}: cleared old results -> {context_size(messages):,}")
        stats["peak_context"] = max(stats["peak_context"], context_size(messages))

        payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
        response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
        response.raise_for_status()
        data = response.json()
        message = data["choices"][0]["message"]
        messages.append(message)
        stats["model_calls"] += 1
        stats["input_tokens"] += (data.get("usage") or {}).get("prompt_tokens", 0)

        calls = message.get("tool_calls") or []
        if not calls:
            return {"status": "done", "summary": message.get("content"), **stats}
        for call in calls:
            name = call["function"]["name"]
            try:
                arguments = json.loads(call["function"]["arguments"] or "{}")
                result = REGISTRY[name](**arguments) if name in REGISTRY else f"Error: unknown tool {name!r}"
            except Exception as error:
                result = f"Error: {type(error).__name__}: {error}"
            result = cap(result)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            stats["tool_calls"] += 1
            short = (call["function"]["arguments"] or "")[:70]
            print(f"[{step:>2}] {name}({short})  ->  {result.splitlines()[0][:60] if result else ''}")
        print(f"     context for next step: about {context_size(messages):,} tokens")

    return {"status": "max_steps", "summary": None, **stats}

## 4. Run it

In [8]:
before = run_command("python -m unittest discover -s tests")
print(before.splitlines()[0], "|", [l for l in before.splitlines() if l.startswith(("FAIL", "Ran", "OK"))])

started = time.perf_counter()
result = run_agent(
    "Customer complaint: 'I've been a customer for five years, and my bill for 10 masala teas was higher than "
    "my friend's, who joined last week. Aren't loyal customers supposed to get a discount?' "
    "Find and fix the bug. The tests are in the tests/ folder."
)
elapsed = time.perf_counter() - started
print("\nStatus: ", result["status"])
print("Summary:", result["summary"])

exit code: 1 | ['FAIL: test_loyal_customers_pay_less (test_pricing.PricingTests.test_loyal_customers_pay_less)', 'Ran 6 tests in 0.001s', 'FAILED (failures=1)']


[ 1] run_command({"command": "python -m unittest discover -s tests"})  ->  exit code: 1
     context for next step: about 1,034 tokens


[ 2] grep({"pattern": "loyal"})  ->  shop/pricing.py:11: def loyalty_discount(amount, years):
     context for next step: about 1,352 tokens


[ 3] find_references({"name": "loyalty_discount"})  ->  shop/pricing.py:26 (in final_price): amount = amount + loyal
     context for next step: about 1,546 tokens


[ 4] read_file({"path": "shop/pricing.py", "start": 22, "end": 29})  ->  shop/pricing.py: lines 22-29 of 29
     context for next step: about 1,679 tokens


[ 5] edit_file({"path": "shop/pricing.py", "old": "amount = amount + loyalty_discount)  ->  Edited shop/pricing.py.
     context for next step: about 1,810 tokens


[ 6] run_command({"command": "python -m unittest discover -s tests"})  ->  exit code: 0
     context for next step: about 1,875 tokens


     MEMORY WRITE: Run tests with: python -m unittest discover -s tests
[ 7] remember({"note": "Run tests with: python -m unittest discover -s tests"})  ->  Saved. Memory now has 2 notes.
     context for next step: about 1,922 tokens



Status:  done
Summary: Loyal customers were charged more because final_price in shop/pricing.py added the loyalty discount instead of subtracting it. I changed + to -, and all tests pass.


## 5. Verify

The same rule as always: our code checks, not the model.

In [9]:
after = run_command("python -m unittest discover -s tests")
passed = after.startswith("exit code: 0")
fixed = (workspace / "shop/pricing.py").read_text()
untouched = all((workspace / p).read_text() == SHOP[p] for p in SHOP if p != "shop/pricing.py")

print("The fix:\n")
for line in difflib.unified_diff(BUGGY.splitlines(), fixed.splitlines(), "pricing.py (before)",
                                 "pricing.py (after)", lineterm="", n=1):
    print(line)

print("\n--- Scorecard ---")
print(f"Tests pass:                   {passed}")
print(f"Only pricing.py changed:      {untouched}")
print(f"Matches the correct code:     {fixed == CORRECT}")
print(f"Model calls / tool calls:     {result['model_calls']} / {result['tool_calls']}")
print(f"Peak context:                 about {result['peak_context']:,} tokens (budget {CONTEXT_BUDGET:,})")
print(f"Times old results cleared:    {result['clearings']}")
print(f"Time:                         {elapsed:.1f}s" + (" (simulated)" if llm.offline else ""))
print("\nMemory after the run:\n" + load_memory())

The fix:

--- pricing.py (before)
+++ pricing.py (after)
@@ -25,3 +25,3 @@
     amount = subtotal(order)
-    amount = amount + loyalty_discount(amount, order.customer.years)
+    amount = amount - loyalty_discount(amount, order.customer.years)
     if coupon:

--- Scorecard ---
Tests pass:                   True
Only pricing.py changed:      True
Matches the correct code:     True
Model calls / tool calls:     8 / 7
Peak context:                 about 1,922 tokens (budget 6,000)
Times old results cleared:    0
Time:                         3.6s (simulated)

Memory after the run:
- Money: prices are in rupees, GST is 18% (saved 2026-11-01)
- Run tests with: python -m unittest discover -s tests (saved 2026-11-13)



In [10]:
# Checkpoint: the offline script must fix the bug exactly, touch nothing else, and stay within budget.
if llm.offline:
    assert passed and untouched and fixed == CORRECT
    assert result["peak_context"] <= CONTEXT_BUDGET
    assert "Run tests with" in load_memory()
print("Checkpoint passed." if passed else "Not solved this time. The trace and scorecard show where it went wrong.")

Checkpoint passed.


"Matches the correct code" is a stricter check than "tests pass". A model could make the tests pass with a different change, such as adjusting the discount rate, that hides the bug instead of fixing it. Here we know the right answer, so we can check for it. In real work you won't, and that's what code review is for.

Notice the path it took: tests, then `grep`, then `find_references`, then eight lines of one file. Search took it from a vague complaint to the one wrong character, and the context stayed well under budget the whole way.

## 6. Run it again

A second report on the same project. This time memory includes how to run the tests, saved during the first run.

In [11]:
(workspace / "shop/pricing.py").write_text(BUGGY)  # the same bug, back again
second = run_agent("The loyalty discount bug is back after a bad merge. Please fix it again.")
print(f"\nFirst run: {result['model_calls']} model calls. Second run: {second['model_calls']}.")

[ 1] run_command({"command": "python -m unittest discover -s tests"})  ->  exit code: 1
     context for next step: about 1,008 tokens


[ 2] grep({"pattern": "loyal"})  ->  shop/pricing.py:11: def loyalty_discount(amount, years):
     context for next step: about 1,326 tokens


[ 3] find_references({"name": "loyalty_discount"})  ->  shop/pricing.py:26 (in final_price): amount = amount + loyal
     context for next step: about 1,520 tokens


[ 4] read_file({"path": "shop/pricing.py", "start": 22, "end": 29})  ->  shop/pricing.py: lines 22-29 of 29
     context for next step: about 1,653 tokens


[ 5] edit_file({"path": "shop/pricing.py", "old": "amount = amount + loyalty_discount)  ->  Edited shop/pricing.py.
     context for next step: about 1,784 tokens


[ 6] run_command({"command": "python -m unittest discover -s tests"})  ->  exit code: 0
     context for next step: about 1,850 tokens


     MEMORY WRITE: Run tests with: python -m unittest discover -s tests
[ 7] remember({"note": "Run tests with: python -m unittest discover -s tests"})  ->  Saved. Memory now has 2 notes.
     context for next step: about 1,897 tokens



First run: 8 model calls. Second run: 8.


The offline script follows the same path every time, so its two numbers match. On a real model, compare the first move of each run. With the test command in memory, a model often skips straight to running the right command instead of trying the default one first.

## You finished Part 3

Your agent can now work on projects bigger than its context. It measures what it uses, finds code instead of reading all of it, sheds old context when it has to, and remembers what it learned.

What it still can't do is **recover when things go wrong** with the model service itself, or **prove** that a change to its prompt made it better. That's Part 4: retries and error handling, tracing, evals, cost control and caching.

## Exercises

1. Lower `CONTEXT_BUDGET` to 2,500 and rerun on a real model. When does clearing start to hurt?
2. Replace clearing with the summarising compaction from notebook 12. Is the summary worth the extra call for this task?
3. Give the agent a task that needs two fixes in two different files. Does it use `find_references` to check the impact of its first fix?

## Clean up

In [12]:
shutil.rmtree(workspace)
print("Removed the temporary workspace.")

Removed the temporary workspace.
